# 产品的单位集中度：分档与分类

基于 `product_unit_matrix_{buy,sell}.csv`（04 的产物），对每个 9 位产品算：

- **首要单位占比** `top1` —— 金额最大的那个单位占该产品金额的比重
- **前 2 个累计** `top2`、**前 3 个累计** `top3`

然后做两件事：

1. 按 **99% / 90% / 80% / 50% / 30%** 分档，统计各档的产品数与金额
2. 把产品分成四类：首要单位 ≥90% ／ 前 2 个 ≥90% ／ 前 3 个 ≥90% ／ 前 3 个仍 <90%

所有输出表都带 `product_name`。

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

pd.set_option('display.max_rows', 80)
pd.set_option('display.width', 200)

# 本 notebook 就放在 profile/ 里；万一在上级目录打开也能找到
PROF = Path.cwd()
if not (PROF / 'product_unit_matrix_buy.csv').exists():
    PROF = PROF / 'profile'
assert (PROF / 'product_unit_matrix_buy.csv').exists(), f'找不到矩阵文件：{PROF}'
print('PROF =', PROF)

_n = pd.read_csv(PROF / 'product_names.csv', dtype=str, encoding='utf-8-sig')
NAME = dict(zip(_n['product_id'].str.strip(), _n['product_name'].str.strip()))
print(f'产品名称 {len(NAME)} 条')

## 1　按产品汇总：top1 / top2 / top3

In [ ]:
TOPK = 5   # 明细表里保留前几个单位

def build(side):
    """每个产品一行：前 TOPK 个单位及其占比，以及 top1/top2/top3 累计占比"""
    m = pd.read_csv(PROF / f'product_unit_matrix_{side}.csv', dtype={'product_id': str})

    # 按产品内金额降序排名
    m = m.sort_values(['product_id', 'value'], ascending=[True, False])
    m['rk'] = m.groupby('product_id').cumcount() + 1

    tot = m.groupby('product_id')['value'].sum()
    m['sh'] = m['value'] / m['product_id'].map(tot)

    g = m[m['rk'] <= TOPK]
    piv_u = g.pivot(index='product_id', columns='rk', values='unit_std')
    piv_s = g.pivot(index='product_id', columns='rk', values='sh')

    d = pd.DataFrame({'product_id': piv_u.index})
    d['product_name'] = d['product_id'].map(NAME).fillna('(无名称)')
    d['v_prod']  = d['product_id'].map(tot)
    d['n_units'] = d['product_id'].map(m.groupby('product_id').size())

    for k in range(1, TOPK + 1):
        d[f'unit{k}']  = d['product_id'].map(piv_u[k]) if k in piv_u.columns else np.nan
        d[f'share{k}'] = d['product_id'].map(piv_s[k]) if k in piv_s.columns else np.nan

    s = [d[f'share{k}'].fillna(0) for k in range(1, 4)]
    d['top1'] = s[0]
    d['top2'] = s[0] + s[1]
    d['top3'] = s[0] + s[1] + s[2]

    return d.sort_values('v_prod', ascending=False).reset_index(drop=True)


D = {side: build(side) for side in ('buy', 'sell')}
for side, d in D.items():
    print(f'{side}: {len(d):,} 个产品,  总金额 {d["v_prod"].sum():.4e}')
display(D['buy'].head(8)[['product_id', 'product_name', 'v_prod', 'n_units',
                          'unit1', 'share1', 'unit2', 'share2', 'unit3', 'share3',
                          'top1', 'top2', 'top3']])

## 2　首要单位占比分档

档位：**≥99% ／ 90–99% ／ 80–90% ／ 50–80% ／ 30–50% ／ <30%**

产品数和金额占比**两个都看**——前面几轮反复踩过坑：不加权的产品计数会被长尾误导，
金额加权才反映对回归的实际影响。

In [ ]:
BINS   = [0, .30, .50, .80, .90, .99, 1.0000001]
LABELS = ['<30%', '30-50%', '50-80%', '80-90%', '90-99%', '>=99%']
ORDER  = LABELS[::-1]          # 展示时从高到低

def bin_table(d, col='top1'):
    x = d.copy()
    x['档位'] = pd.cut(x[col], bins=BINS, labels=LABELS, right=False)
    g = x.groupby('档位', observed=False).agg(产品数=('product_id', 'size'),
                                              金额=('v_prod', 'sum'))
    g = g.reindex(ORDER)
    g['产品数占比%'] = (g['产品数'] / g['产品数'].sum() * 100).round(2)
    g['金额占比%']  = (g['金额']  / g['金额'].sum()  * 100).round(2)
    g['累计产品数%'] = g['产品数占比%'].cumsum().round(2)
    g['累计金额%']  = g['金额占比%'].cumsum().round(2)
    g['金额'] = g['金额'].map(lambda v: f'{v:.4e}')
    return g[['产品数', '产品数占比%', '累计产品数%', '金额', '金额占比%', '累计金额%']]

for side in ('buy', 'sell'):
    print(f'\n===== {side} 侧　首要单位占比 top1 分档 =====')
    display(bin_table(D[side], 'top1'))

In [ ]:
# 同样的档位，看 top2 / top3 累计占比 —— 用来判断"放宽到前2/前3个单位"能救回多少
for side in ('buy', 'sell'):
    for col, lab in [('top2', '前 2 个单位累计'), ('top3', '前 3 个单位累计')]:
        print(f'\n===== {side} 侧　{lab} 分档 =====')
        display(bin_table(D[side], col))

## 3　产品四分类

互斥分类，逐级放宽：

| 类 | 定义 |
|---|---|
| **A** | 首要单位 ≥ 90% |
| **B** | 首要单位 < 90%，但前 **2** 个加起来 ≥ 90% |
| **C** | 前 2 个 < 90%，但前 **3** 个加起来 ≥ 90% |
| **D** | 前 3 个加起来仍 < 90% |

In [ ]:
CLS = ['A 首要单位>=90%', 'B 前2个>=90%', 'C 前3个>=90%', 'D 前3个仍<90%']

def add_class(d, thr=0.90):
    d = d.copy()
    d['class'] = np.select(
        [d['top1'] >= thr, d['top2'] >= thr, d['top3'] >= thr],
        CLS[:3], default=CLS[3])
    return d

for side in ('buy', 'sell'):
    D[side] = add_class(D[side])

def class_table(d):
    g = d.groupby('class', observed=False).agg(产品数=('product_id', 'size'),
                                               金额=('v_prod', 'sum'),
                                               平均单位数=('n_units', 'mean'))
    g = g.reindex(CLS)
    g['产品数占比%'] = (g['产品数'] / g['产品数'].sum() * 100).round(2)
    g['金额占比%']  = (g['金额']  / g['金额'].sum()  * 100).round(2)
    g['累计金额%']  = g['金额占比%'].cumsum().round(2)
    g['平均单位数'] = g['平均单位数'].round(1)
    g['金额'] = g['金额'].map(lambda v: f'{v:.4e}')
    return g[['产品数', '产品数占比%', '金额', '金额占比%', '累计金额%', '平均单位数']]

for side in ('buy', 'sell'):
    print(f'\n===== {side} 侧　产品四分类 =====')
    display(class_table(D[side]))

## 4　导出（都带产品名）

In [ ]:
COLS = (['product_id', 'product_name', 'class', 'v_prod', 'n_units',
         'top1', 'top2', 'top3'] +
        [c for k in range(1, TOPK + 1) for c in (f'unit{k}', f'share{k}')])

for side in ('buy', 'sell'):
    d = D[side][COLS]
    f = PROF / f'product_concentration_{side}.csv'
    d.to_csv(f, index=False, encoding='utf-8-sig')
    print(f'{f.name}: {len(d):,} 行')

    for i, cls in enumerate(CLS[:3], start=1):
        sub = d[d['class'] == cls]
        g = PROF / f'class_{chr(64 + i)}_{side}.csv'
        sub.to_csv(g, index=False, encoding='utf-8-sig')
        print(f'   {g.name}: {len(sub):,} 个产品,  金额占比 '
              f'{sub["v_prod"].sum() / d["v_prod"].sum() * 100:.2f}%')

## 5　看看每一类具体长什么样

In [ ]:
side = 'buy'
SHOW = ['product_id', 'product_name', 'v_prod', 'n_units',
        'unit1', 'share1', 'unit2', 'share2', 'unit3', 'share3', 'top1', 'top2', 'top3']

for cls in CLS:
    sub = D[side][D[side]['class'] == cls].head(10)
    print(f'\n===== {side}　{cls}　（金额最大的 10 个）=====')
    display(sub[SHOW].reset_index(drop=True))